# 01 — Resumable Ninisite Scraping

## What this notebook is teaching

The important idea is **not the syntax of `requests`, BeautifulSoup, or CSV writing**. The important idea is how to design a web-acquisition system that can survive an unreliable source.

Ninisite frequently throttled repeated requests, returned empty pages, or interrupted long scraping runs. A one-shot scraper would therefore be fragile: if it failed after many hours, the whole run could be lost or restarted from the beginning.

The solution used here is a **resumable stateful scraping algorithm**:

1. discover thread identifiers and URLs;
2. scrape each thread page-by-page;
3. write successfully recovered data to disk immediately;
4. remember which thread IDs are complete;
5. treat empty/throttled results as unresolved rather than as valid empty threads;
6. on the next run, skip completed threads and retry only unresolved work;
7. run recovery passes until unresolved work is approximately zero.

This is the same engineering idea used in fault-tolerant batch systems: **progress is externalized as durable state**.

### Key concepts demonstrated

The notebook demonstrates:

- why checkpoints are necessary;
- why an empty HTTP result is not automatically evidence that a thread contains no posts;
- why retries use increasing waiting time;
- why the thread manifest is separated from downloaded posts;
- why successful work is skipped on reruns;
- why parsing and transport are separate concerns.

Exact HTML selectors and Python syntax are implementation details; the methodological focus is the recovery strategy and data contract.


In [ ]:
from pathlib import Path
import sys, json
HERE = Path.cwd().resolve()
ROOT = HERE.parent if HERE.name == "notebooks" else HERE
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.common import load_json, set_seed
CFG = load_json(ROOT / "configs" / "project_config.json")
LEX = load_json(ROOT / "configs" / "lexicons.json")
set_seed(CFG["random_seed"])
print("Project root:", ROOT)
print("Project:", CFG["project_name"])

## Category metadata is assigned at acquisition time

The historical project exposed a useful failure mode: a few sub-category source CSVs
were scraped successfully but `category` / `sub_category` were not written into the
rows. The content was recoverable only because the source filenames were still known.

The final scraper prevents this by treating section identity as part of the input
manifest. Every row inherits `category` and `sub_category` from the current
`forum_sections.csv` entry **before it is checkpointed**. Therefore combining scraper
outputs later cannot lose section provenance.

This follows a general data-engineering rule: metadata known at acquisition time should
be persisted immediately rather than reconstructed downstream.


In [ ]:
import pandas as pd
from src.scraping import parse_forum_page, parse_topic_page, collect_thread_manifest, scrape_manifest_resumable, recovery_pass, scrape_profiles_resumable
SCFG = CFG["scraping"]
RUN_LIVE = bool(CFG.get("run_live_scrape", False))
print("RUN_LIVE =", RUN_LIVE)

## 1. Parser smoke test — separate parsing from networking

A scraper has two different problems:

- **transport:** obtaining an HTML page reliably;
- **parsing:** converting that HTML into structured fields.

Keeping those problems separate makes debugging much easier. The parser can be tested on a small local HTML example even when the website is unavailable.

### Algorithmic idea

For a forum listing page:

`HTML → locate topic links → extract thread ID/title/reply count → structured records`

For a topic page:

`HTML → locate each post container → extract author/time/content/likes/post ID → structured records`

BeautifulSoup is used because the source is nested HTML rather than a stable API. It lets the scraper reason about the document tree instead of relying on brittle string slicing.


In [ ]:
sample_forum = """<html><a href="/discussion/topic/123/test"><span class="topic_subject">Demo topic</span></a><span class="topic_number">4</span></html>"""
sample_topic = """<html><article class="topic-post" id="post-456"><a class="nickname" href="/user/demo"><span itemprop="name">کاربر</span></a><span class="date">1404/01/01</span><span class="time">10:00</span><div class="post-count">تعداد پست: 12</div><div class="post-message">خیلی نگرانم و استرس دارم</div><a class="like-count" data-like-count="2"></a></article></html>"""
print(parse_forum_page(sample_forum, SCFG["base_domain"]))
print(parse_topic_page(sample_topic, 1, SCFG["base_domain"]))

## 2. Load the forum-section configuration

The forum URLs are data, not program logic, so they live in `inputs/forum_sections.csv`.

That separation matters for reproducibility: the scraping algorithm stays unchanged when a different Ninisite section is collected. Only the configuration changes.

Live scraping is disabled by default because repeated requests affect an external service. The notebook can therefore be executed safely in demonstration/reproduction mode without contacting the site.


In [ ]:
sections = pd.read_csv(ROOT / SCFG["sections_file"], comment="#")
display(sections)
if RUN_LIVE and (sections.empty or sections["forum_url"].astype(str).str.contains("REPLACE_ME").any()):
    raise ValueError("Edit inputs/forum_sections.csv before enabling live scraping.")

## 3. Build the thread manifest

The **thread manifest** is the acquisition plan. It is intentionally created before downloading all posts.

Conceptually it contains:

`thread_id → thread_url → title → section → listing page`

Why use a manifest?

1. It gives a finite list of expected work.
2. It lets us compare **expected threads** with **successfully recovered threads**.
3. It prevents a failed request from silently disappearing.
4. It makes recovery targeted rather than repeating the entire scrape.

The key system-design invariant is:

> Every thread in the manifest must eventually be either recovered successfully or remain explicitly unresolved.

A missing thread should never be mistaken for a successfully empty thread.


In [ ]:
manifest_path = ROOT / SCFG["threads_manifest"]
if RUN_LIVE:
    manifest = collect_thread_manifest(sections, SCFG)
    manifest.to_csv(manifest_path, index=False, encoding="utf-8-sig")
elif manifest_path.exists():
    manifest = pd.read_csv(manifest_path, dtype=str)
else:
    manifest = pd.DataFrame(columns=["thread_id","thread_title","thread_url","reply_count","section_name","forum_page"])
print("Manifest rows:", len(manifest))
display(manifest.head())

## 4. Main resumable thread scraper

This is the core fault-tolerant algorithm.

### State transition

For each thread:

`PENDING → REQUESTING → SUCCESS → CHECKPOINTED`

or, after a temporary failure:

`PENDING → REQUESTING → RETRY → ...`

If all retries fail:

`PENDING → UNRESOLVED`

A later notebook run begins by reading the checkpoint and **does not request already completed threads again**.

### Why backoff?

When a server responds poorly because requests are arriving too quickly, immediately retrying can make the problem worse. Increasing the delay between attempts gives the server time to recover and reduces the chance of escalating throttling.

A simplified retry schedule is:

`wait = base_delay × attempt_number`

The exact number of seconds is a tunable operational parameter; the algorithmic idea is **progressively gentler retrying**.

### Why checkpoint after success?

RAM is temporary. CSV/checkpoint state survives a crash, notebook restart, network loss, or IP throttling. Persisting successful work converts a long scrape into many small recoverable transactions.

The historical project required several reruns for exactly this reason.


In [ ]:
if RUN_LIVE and len(manifest):
    posts = scrape_manifest_resumable(manifest, SCFG, ROOT)
else:
    checkpoint = ROOT / SCFG["posts_checkpoint"]
    posts = pd.read_csv(checkpoint, low_memory=False) if checkpoint.exists() else pd.DataFrame()
print("Checkpointed posts:", len(posts))

## 5. Recovery pass — measure what is still missing

After the main pass, the algorithm does not assume the scrape is complete. It computes:

`unresolved_threads = manifest_thread_ids − recovered_thread_ids`

Only this unresolved set is retried.

This is an important difference from blindly rerunning the whole scraper. Recovery cost becomes proportional to the remaining failures rather than to the entire forum.

The cell is intentionally rerunnable across sessions:

`recover → checkpoint → recompute missing set → recover again`

until the unresolved count is acceptably close to zero.

### Why this matters scientifically

If failed pages are silently dropped, the dataset can acquire an accidental sampling bias: content that is harder to retrieve becomes systematically absent. Explicit missing-work accounting makes that problem visible.


In [ ]:
if RUN_LIVE and len(manifest):
    remaining = None
    for round_no in range(int(SCFG.get("recovery_rounds_per_run", 1))):
        posts, remaining = recovery_pass(manifest, SCFG, ROOT)
        print(f"Recovery round {round_no+1}: unresolved = {len(remaining)}")
        if len(remaining) == 0: break
else:
    remaining = manifest.copy() if len(manifest) else pd.DataFrame()
print("Unresolved threads now:", len(remaining))

## 6. Profile enrichment — including the gender-parser lesson

Profile metadata is collected as a separate resumable stage.

### What went wrong in the original development attempt

The first gender parser assumed the wrong HTML marker semantics. As a result,
many profiles were left unresolved. Later, after `profile_url` had already
been removed from an intermediate cleaned table, the team had to reconstruct
the URL using:

`thread_id + author + posted_at → original combined_all.csv → profile_url`

and selectively re-scrape only those unresolved profiles.

This was a **development repair**, not the intended final architecture.

### How the final scraper prevents the same problem

The current parser recognizes both the newer and older marker families:

- female evidence: `is-woman`, `iconwoman`, related woman markers;
- male evidence: `is-user`, `iconuser-01` (the marker shown in the historical notebook), `iconuser-1` as a tolerated alias, `iconman`, and related user/man markers;
- explicit nearby Persian text (`زن` or `مرد`) takes precedence when present.

It never uses the unsafe rule “not female = male”.

If the HTML is ambiguous, the result remains `unknown / -1`. Such a profile is
**not considered complete for gender purposes**, so the scraping stage may retry
it in a later recovery pass. The checkpoint is upserted by `profile_url`, so a
later successful retry replaces the unresolved result rather than creating a
duplicate.

This means future runs solve gender uncertainty in the **scraping stage**, where
the profile URL still naturally exists, instead of pushing network work into
data cleaning.


In [ ]:
from src.scraping import detect_gender_from_profile_html

female_html = '<div class="profile"><i class="is-woman"></i><span>زن</span></div>'
male_html   = '<div class="profile"><i class="is-user"></i><span>مرد</span></div>'
unknown_html = '<div class="profile"><span>اطلاعات کاربر</span></div>'

assert detect_gender_from_profile_html(female_html)[:2] == ("زن", 1)
assert detect_gender_from_profile_html(male_html)[:2] == ("مرد", 0)
assert detect_gender_from_profile_html(unknown_html)[:2] == ("unknown", -1)

print("Gender parser checks passed:")
print(" female:", detect_gender_from_profile_html(female_html))
print(" male:  ", detect_gender_from_profile_html(male_html))
print(" unknown:", detect_gender_from_profile_html(unknown_html))


In [ ]:
if RUN_LIVE and len(posts):
    profiles = scrape_profiles_resumable(posts, SCFG, ROOT)
else:
    pp = ROOT / SCFG["profile_checkpoint"]
    profiles = pd.read_csv(pp, low_memory=False) if pp.exists() else pd.DataFrame()
print("Profile rows:", len(profiles))

## 7. Completion summary and invariants

The final counts are not just logging. They are a quick integrity check:

- manifest thread count = expected acquisition workload;
- checkpoint post count = durable recovered content;
- unresolved thread count = remaining scraping debt;
- profile row count = recovered profile metadata.

For a real run, the desired state is **unresolved threads ≈ 0**, not necessarily literally zero if a very small number of pages are permanently inaccessible.

### System-designer takeaway

The scraping algorithm is best described as:

**manifest-driven + retry-aware + checkpointed + resumable + auditable.**

The project did not depend on one perfect continuous scraping session.


In [ ]:
summary = {"manifest_threads":len(manifest),"checkpoint_posts":len(posts),"unresolved_threads":len(remaining),"profile_rows":len(profiles),"live_scrape_enabled":RUN_LIVE}
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("SCRAPING NOTEBOOK COMPLETED")